# Concaténation des données météo — 12 mois les plus récents

Lit tous les fichiers `data/weather/historique-meteo-toulouse-*.csv`, sélectionne les 12 mois les plus récents disponibles et les concatène en un seul DataFrame.

In [ ]:
import pandas as pd
import glob
import os
import re

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'data', 'weather')
# Si le notebook est lancé depuis script/weather, remonter d'un niveau
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join('..', '..', 'data', 'weather')

pattern = os.path.join(DATA_DIR, 'historique-meteo-toulouse-*.csv')
files = sorted(glob.glob(pattern))
print(f"{len(files)} fichier(s) trouvé(s) dans {DATA_DIR}")
for f in files:
    print(' ', os.path.basename(f))

In [ ]:
def extract_year_month(path):
    m = re.search(r'(\d{4})-(\d{2})\.csv$', path)
    if m:
        return int(m.group(1)), int(m.group(2))
    return (0, 0)

files_sorted = sorted(files, key=extract_year_month)
recent_12 = files_sorted[-12:]

print("12 mois les plus récents sélectionnés :")
for f in recent_12:
    print(' ', os.path.basename(f))

In [ ]:
def read_meteo_csv(path):
    # Les 3 premières lignes sont des commentaires, la 4e est l'en-tête
    df = pd.read_csv(path, skiprows=3, parse_dates=['DATE'])
    return df

frames = [read_meteo_csv(f) for f in recent_12]
df = pd.concat(frames, ignore_index=True)
df = df.sort_values('DATE').reset_index(drop=True)

print(f"Période : {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"Nombre de lignes : {len(df)}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
output_path = os.path.join(DATA_DIR, 'meteo_toulouse_12_mois.csv')
df.to_csv(output_path, index=False)
print(f"Fichier enregistré : {output_path}")

Clean Meteo Codes

In [ ]:
df_code = pd.read_csv(DATA_DIR + '/Codes meteo.csv')
df_code = df_code[["CodeMétéo", "Condition"]]
df_code.to_csv(os.path.join(DATA_DIR, 'meteo_toulouse_codes.csv'), index=False)